# Data analysis agent with sandbox tools

This notebook builds a [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) agent that analyses a CSV file in a prokube sandbox and shows a chart. The agent's tools are plain functions around `SandboxClient`, and the notebook owns the sandbox: it creates it, resumes it for each question, suspends it after each answer and deletes it at the end. See the [README](./README.md) for running it on a laptop and for the same tools in another framework.

Sandboxes must be enabled on your prokube cluster, and the Lab needs sandbox access.

## Setup

Install the prokube SDK and LangChain:

In [ ]:
%pip install -q "prokube-sdk @ git+https://github.com/prokube/prokube-sdk@sandboxv3/main" "langchain>=1,<2" "langchain-openai>=1,<2"

In a Lab with sandbox access, `SandboxClient()` needs no configuration: it uses the Lab's workspace and identity.

In [ ]:
from prokube.sandbox import SandboxClient

sandboxes = SandboxClient()
SANDBOX = "data-analysis"

## Choose a model

Any OpenAI-compatible endpoint with tool calling works, for example a model from prokube LLM Serving or an external provider. The cell uses `OPENAI_BASE_URL`, `OPENAI_MODEL` and `OPENAI_API_KEY` if they are set:

In [ ]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL") or input("Base URL: "),
    model=os.environ.get("OPENAI_MODEL") or input("Model: "),
    api_key=os.environ.get("OPENAI_API_KEY") or getpass("API key: "),
)

## The tools

Each tool is a few lines around the SDK. The agent only sees their names, signatures and docstrings.

In [ ]:
from IPython.display import Image, display


def run_python(code: str) -> str:
    """Run a Python script in the sandbox, in /workspace. pandas and matplotlib are installed."""
    result = sandboxes.run(SANDBOX, "/workspace/venv/bin/python -", stdin=code, cwd="/workspace")
    return f"exit code {result.exit_code}\n{result.stdout}{result.stderr}"


def write_file(path: str, content: str) -> str:
    """Write a text file under /workspace in the sandbox."""
    sandboxes.write_file(SANDBOX, path, content)
    return f"Wrote {path}"


def read_file(path: str) -> str:
    """Read a text file under /workspace in the sandbox."""
    return sandboxes.read_file(SANDBOX, path).decode()


def show_image(path: str) -> str:
    """Show a PNG file from the sandbox to the user."""
    display(Image(sandboxes.read_file(SANDBOX, path)))
    return f"Showed {path}"

## The agent

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools=[run_python, write_file, read_file, show_image],
    system_prompt=(
        "You are a data analyst. Use the tools to work in the sandbox. "
        "Save charts as PNG files in /workspace and show them. Answer briefly."
    ),
    checkpointer=InMemorySaver(),  # keeps the conversation across questions
)

## Create the sandbox

Create the sandbox, install pandas and matplotlib into `/workspace` and upload the CSV. `/workspace` survives suspend and resume, so this runs once per conversation:

In [ ]:
import time

start = time.perf_counter()
sandboxes.create(SANDBOX)
print(f"Created the sandbox in {time.perf_counter() - start:.2f} s")

sandboxes.run(
    SANDBOX,
    "python3 -m venv /workspace/venv && /workspace/venv/bin/pip install -q pandas matplotlib",
    timeout=300,
)
with open("sales.csv") as f:
    sandboxes.write_file(SANDBOX, "/workspace/sales.csv", f.read())
sandboxes.suspend(SANDBOX)

`ask` resumes the sandbox, lets the agent answer and suspends the sandbox again, so it uses no resources between questions:

In [ ]:
conversation = {"configurable": {"thread_id": "sales"}}


def ask(question: str) -> str:
    start = time.perf_counter()
    sandboxes.resume(SANDBOX)
    print(f"Resumed the sandbox in {time.perf_counter() - start:.2f} s")
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, conversation)
    finally:
        sandboxes.suspend(SANDBOX)
    return result["messages"][-1].content

## Ask

In [ ]:
print(ask("Which region grew fastest in /workspace/sales.csv? Plot monthly revenue per region."))

The agent keeps the conversation, and the sandbox keeps its files:

In [ ]:
print(ask("Add a line for the total to the same chart and show it again."))

## Clean up

Delete the sandbox and its `/workspace`:

In [ ]:
sandboxes.delete(SANDBOX)
sandboxes.close()